In [2]:
!pip install --quiet langchain langchain-core langchain-community langchain-groq datasets tqdm

print("✅ Packages installed! Restart runtime if needed, then run Cell 33B.")

✅ Packages installed! Restart runtime if needed, then run Cell 33B.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import userdata
import os
import json
from tqdm import tqdm

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
print(f"✅ GROQ API Key loaded: {os.environ['GROQ_API_KEY'][:10]}...")

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

print("✅ All imports successful!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ GROQ API Key loaded: gsk_L0c2n9...
✅ All imports successful!


In [4]:
# ============================================================================
# CELL 33C: MAIN EVALUATION CODE
# ============================================================================

# Configuration
DOMAINS_TO_EVALUATE = [
    'medical_covidqa',
    'finance_finqa',
    'support_techqa',
    'general_hotpotqa',
    'legal_cuad',
]

ANSWERS_BASE_DIR = "/content/drive/MyDrive/ragbench_v2/results/final_multi_domain_answers"
EVAL_OUTPUT_DIR = "/content/drive/MyDrive/ragbench_v2/results/evaluation_with_llm_judge"
os.makedirs(EVAL_OUTPUT_DIR, exist_ok=True)

EVAL_LLM_MODEL = "llama-3.3-70b-versatile"
EVAL_LLM_TEMP = 0

print(f"{'='*80}")
print(f"📊 RAGBENCH EVALUATION WITH LLM JUDGE")
print(f"{'='*80}")
print(f"Evaluator model: {EVAL_LLM_MODEL}")
print(f"Domains: {len(DOMAINS_TO_EVALUATE)}")
print(f"Input: {ANSWERS_BASE_DIR}")
print(f"Output: {EVAL_OUTPUT_DIR}")
print(f"{'='*80}\n")

# Initialize LLM
eval_llm = ChatGroq(
    model=EVAL_LLM_MODEL,
    temperature=EVAL_LLM_TEMP,
    max_tokens=8192
)
print(f"✅ Initialized evaluation LLM: {EVAL_LLM_MODEL}\n")

# Evaluation Prompt
EVALUATION_PROMPT = """You are an expert evaluator analyzing the quality and attribution of a RAG (Retrieval-Augmented Generation) system's response.

Given:
1. Question: {question}
2. Generated Answer: {answer}
3. Retrieved Documents: {retrieved_docs}

Analyze the response and provide a detailed evaluation in JSON format with the following fields:

1. "relevance_explanation": Explain which documents are relevant to answering the question and why.

2. "all_relevant_sentence_keys": List all document IDs or sentence keys that contain information relevant to the question.

3. "overall_supported_explanation": Detailed explanation of whether the answer is supported by the retrieved documents. Analyze each claim in the answer.

4. "overall_supported": Boolean - is the entire answer supported by the documents?

5. "sentence_support_information": Array of objects, one for each sentence in the answer:
   - "response_sentence_key": Label for the sentence (a, b, c, etc.)
   - "explanation": Why this sentence is or isn't supported
   - "supporting_sentence_keys": Array of document IDs that support this sentence
   - "fully_supported": Boolean - is this sentence fully supported?

6. "all_utilized_sentence_keys": List of all document IDs actually used in the answer.

Return ONLY valid JSON, no additional text.

Example format:
{{
  "relevance_explanation": "Document X contains...",
  "all_relevant_sentence_keys": ["doc1", "doc2"],
  "overall_supported_explanation": "The answer claims... which is supported by...",
  "overall_supported": true,
  "sentence_support_information": [
    {{
      "response_sentence_key": "a",
      "explanation": "This sentence...",
      "supporting_sentence_keys": ["doc1"],
      "fully_supported": true
    }}
  ],
  "all_utilized_sentence_keys": ["doc1", "doc2"]
}}"""

# Helper Functions
def load_question_data(question_id: str, domain: str) -> dict:
    base_path = os.path.join(ANSWERS_BASE_DIR, domain)

    answer_file = os.path.join(base_path, f"{question_id}_answer.txt")
    answer = None
    if os.path.exists(answer_file):
        with open(answer_file, 'r', encoding='utf-8') as f:
            content = f.read()
            lines = content.split('\n')
            answer_start = next((i for i, line in enumerate(lines) if line.startswith('Answer:')), None)
            if answer_start is not None:
                answer = '\n'.join(lines[answer_start + 1:]).strip()

    structured_file = os.path.join(base_path, f"{question_id}_structured_output.txt")
    structured = None
    if os.path.exists(structured_file):
        try:
            with open(structured_file, 'r', encoding='utf-8') as f:
                structured = json.load(f)
        except:
            structured = None

    docs_file = os.path.join(base_path, f"{question_id}_retrieved_documents.json")
    retrieved_docs = None
    if os.path.exists(docs_file):
        try:
            with open(docs_file, 'r', encoding='utf-8') as f:
                retrieved_docs = json.load(f)
        except:
            retrieved_docs = None

    return {
        'answer': answer,
        'structured': structured,
        'retrieved_docs': retrieved_docs,
        'files_exist': all([answer, structured, retrieved_docs])
    }

def format_retrieved_docs_for_eval(retrieved_docs: list) -> str:
    formatted = []
    for i, doc in enumerate(retrieved_docs):
        if isinstance(doc, dict):
            doc_id = doc.get('metadata', {}).get('chunk_index', f'doc_{i}')
            content = doc.get('page_content', doc.get('content', ''))
        else:
            doc_id = f'doc_{i}'
            content = str(doc)
        formatted.append(f"[{doc_id}]\n{content}\n")
    return "\n".join(formatted)

def evaluate_answer_with_llm(question: str, answer: str, retrieved_docs: list) -> dict:
    docs_text = format_retrieved_docs_for_eval(retrieved_docs)
    prompt = ChatPromptTemplate.from_template(EVALUATION_PROMPT)
    parser = JsonOutputParser()
    chain = prompt | eval_llm | parser

    try:
        evaluation = chain.invoke({
            "question": question,
            "answer": answer,
            "retrieved_docs": docs_text
        })
        return evaluation
    except Exception as e:
        print(f"    ⚠️  Evaluation failed: {str(e)[:100]}")
        return None

# Main Evaluation Loop
print(f"{'='*80}")
print(f"🚀 STARTING EVALUATION")
print(f"{'='*80}\n")

evaluation_stats = {}

for domain_key in DOMAINS_TO_EVALUATE:
    print(f"\n{'─'*80}")
    print(f"📊 Domain: {domain_key.upper()}")
    print(f"{'─'*80}")

    domain_dir = os.path.join(ANSWERS_BASE_DIR, domain_key)
    if not os.path.exists(domain_dir):
        print(f"⚠️  Directory not found: {domain_dir}")
        continue

    answer_files = [f for f in os.listdir(domain_dir) if f.endswith('_answer.txt')]
    question_ids = [f.replace('_answer.txt', '') for f in answer_files]
    print(f"Found {len(question_ids)} questions")

    domain_eval_dir = os.path.join(EVAL_OUTPUT_DIR, domain_key)
    os.makedirs(domain_eval_dir, exist_ok=True)

    successful = 0
    failed = 0
    skipped = 0

    for question_id in tqdm(question_ids, desc=f"Evaluating {domain_key}"):
        eval_file = os.path.join(domain_eval_dir, f"{question_id}_evaluation.json")
        if os.path.exists(eval_file):
            skipped += 1
            successful += 1
            continue

        data = load_question_data(question_id, domain_key)
        if not data['files_exist']:
            failed += 1
            continue

        question = data['structured'].get('question', '')
        answer = data['answer']
        retrieved_docs = data['retrieved_docs']

        evaluation = evaluate_answer_with_llm(question, answer, retrieved_docs)
        if evaluation is None:
            failed += 1
            continue

        evaluation['_metadata'] = {
            'model': EVAL_LLM_MODEL,
            'question_id': question_id,
            'domain': domain_key,
        }

        with open(eval_file, 'w', encoding='utf-8') as f:
            json.dump(evaluation, f, indent=2, ensure_ascii=False)

        successful += 1

    evaluation_stats[domain_key] = {
        'total': len(question_ids),
        'successful': successful,
        'failed': failed,
        'skipped': skipped,
        'success_rate': successful / len(question_ids) * 100 if len(question_ids) > 0 else 0
    }

    print(f"\n✅ {domain_key}: {successful}/{len(question_ids)} evaluated")
    print(f"   (Skipped {skipped} already done, {failed} failed)")

# Save Summary
summary = {
    'evaluator_model': EVAL_LLM_MODEL,
    'domains_evaluated': list(evaluation_stats.keys()),
    'total_evaluations': sum(s['successful'] for s in evaluation_stats.values()),
    'total_failed': sum(s['failed'] for s in evaluation_stats.values()),
    'per_domain_stats': evaluation_stats
}

summary_file = os.path.join(EVAL_OUTPUT_DIR, 'evaluation_summary.json')
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n{'='*80}")
print(f"📊 EVALUATION COMPLETE")
print(f"{'='*80}")
print(f"Total evaluations: {summary['total_evaluations']}")
print(f"Failed: {summary['total_failed']}")
print(f"\nPer-domain breakdown:")
for domain, stats in evaluation_stats.items():
    print(f"  {domain}: {stats['successful']}/{stats['total']} ({stats['success_rate']:.1f}%)")
print(f"\n✅ Now run RMSE calculation!")

📊 RAGBENCH EVALUATION WITH LLM JUDGE
Evaluator model: llama-3.3-70b-versatile
Domains: 5
Input: /content/drive/MyDrive/ragbench_v2/results/final_multi_domain_answers
Output: /content/drive/MyDrive/ragbench_v2/results/evaluation_with_llm_judge

✅ Initialized evaluation LLM: llama-3.3-70b-versatile

🚀 STARTING EVALUATION


────────────────────────────────────────────────────────────────────────────────
📊 Domain: MEDICAL_COVIDQA
────────────────────────────────────────────────────────────────────────────────
Found 30 questions


Evaluating medical_covidqa: 100%|██████████| 30/30 [00:00<00:00, 96.42it/s]



✅ medical_covidqa: 30/30 evaluated
   (Skipped 30 already done, 0 failed)

────────────────────────────────────────────────────────────────────────────────
📊 Domain: FINANCE_FINQA
────────────────────────────────────────────────────────────────────────────────
Found 30 questions


Evaluating finance_finqa: 100%|██████████| 30/30 [01:18<00:00,  2.62s/it]



✅ finance_finqa: 30/30 evaluated
   (Skipped 0 already done, 0 failed)

────────────────────────────────────────────────────────────────────────────────
📊 Domain: SUPPORT_TECHQA
────────────────────────────────────────────────────────────────────────────────
Found 30 questions


...


✅ support_techqa: 30/30 evaluated
   (Skipped 0 already done, 0 failed)

────────────────────────────────────────────────────────────────────────────────
📊 Domain: GENERAL_HOTPOTQA
────────────────────────────────────────────────────────────────────────────────
Found 30 questions


Evaluating general_hotpotqa: 100%|██████████| 30/30 [01:08<00:00,  2.29s/it]



✅ general_hotpotqa: 30/30 evaluated
   (Skipped 0 already done, 0 failed)

────────────────────────────────────────────────────────────────────────────────
📊 Domain: LEGAL_CUAD
────────────────────────────────────────────────────────────────────────────────
Found 25 questions


Evaluating legal_cuad: 100%|██████████| 25/25 [01:04<00:00,  2.58s/it]



✅ legal_cuad: 25/25 evaluated
   (Skipped 0 already done, 0 failed)

📊 EVALUATION COMPLETE
Total evaluations: 145
Failed: 0

Per-domain breakdown:
  medical_covidqa: 30/30 (100.0%)
  finance_finqa: 30/30 (100.0%)
  support_techqa: 30/30 (100.0%)
  general_hotpotqa: 30/30 (100.0%)
  legal_cuad: 25/25 (100.0%)

✅ Now run RMSE calculation!


In [5]:
# ============================================================================
# RMSE CALCULATION
# ============================================================================

import numpy as np
import pandas as pd
from datasets import load_dataset
import json
import os

# Configuration
DOMAINS = [
    "medical_covidqa",
    "finance_finqa",
    "support_techqa",
    "general_hotpotqa",
    "legal_cuad"
]

DOMAIN_MAPPING = {
    "medical_covidqa": "covidqa",
    "finance_finqa": "finqa",
    "support_techqa": "techqa",
    "general_hotpotqa": "hotpotqa",
    "legal_cuad": "cuad"
}

RESULTS_BASE_DIR = "/content/drive/MyDrive/ragbench_v2/results/final_multi_domain_answers"
EVALUATION_DIR = "/content/drive/MyDrive/ragbench_v2/results/evaluation_with_llm_judge"
OUTPUT_DIR = "/content/drive/MyDrive/ragbench_v2/results/rmse_corrected"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("="*80)
print("RMSE CALCULATION")
print("="*80)

# Load Ground Truth
def load_ground_truth(dataset_name):
    dataset = load_dataset("galileo-ai/ragbench", dataset_name, split="test")
    records = []
    for row in dataset:
        records.append({
            "id": str(row["id"]),
            "relevance_score_gt": row.get("relevance_score", np.nan),
            "utilization_score_gt": row.get("utilization_score", np.nan),
            "completeness_score_gt": row.get("completeness_score", np.nan),
            "adherence_score_gt": float(row.get("adherence_score", 0)),
        })
    return pd.DataFrame(records)

# Load LLM Evaluations and Calculate Predictions
def load_evaluations_and_calc_predictions(domain):
    eval_dir = os.path.join(EVALUATION_DIR, domain)
    results_dir = os.path.join(RESULTS_BASE_DIR, domain)

    predictions = []
    eval_files = [f for f in os.listdir(eval_dir) if f.endswith('_evaluation.json')]

    for eval_file in eval_files:
        q_id = eval_file.replace('_evaluation.json', '')

        # Load evaluation
        with open(os.path.join(eval_dir, eval_file), 'r') as f:
            evaluation = json.load(f)

        # Load retrieved docs to get count
        docs_file = os.path.join(results_dir, f"{q_id}_retrieved_documents.json")
        with open(docs_file, 'r') as f:
            retrieved_docs = json.load(f)

        num_retrieved = len(retrieved_docs)
        relevant_keys = set(evaluation.get("all_relevant_sentence_keys", []))
        utilized_keys = set(evaluation.get("all_utilized_sentence_keys", []))
        adherence = float(evaluation.get("overall_supported", False))

        # Calculate scores
        relevance_score = len(relevant_keys) / num_retrieved if num_retrieved > 0 else 0.0
        utilization_score = len(utilized_keys) / num_retrieved if num_retrieved > 0 else 0.0
        relevant_utilized = relevant_keys.intersection(utilized_keys)
        completeness_score = len(relevant_utilized) / len(relevant_keys) if len(relevant_keys) > 0 else 0.0

        predictions.append({
            "id": q_id,
            "relevance_score_pred": relevance_score,
            "utilization_score_pred": utilization_score,
            "completeness_score_pred": completeness_score,
            "adherence_score_pred": adherence
        })

    return pd.DataFrame(predictions)

# Calculate RMSE
def calc_rmse(pred, gt):
    valid = ~(np.isnan(pred) | np.isnan(gt))
    if not np.any(valid):
        return {"rmse": np.nan, "mae": np.nan, "n": 0}
    p, g = pred[valid], gt[valid]
    return {
        "rmse": float(np.sqrt(np.mean((p - g) ** 2))),
        "mae": float(np.mean(np.abs(p - g))),
        "bias": float(np.mean(p - g)),
        "n": int(len(p)),
        "corr": float(np.corrcoef(p, g)[0, 1]) if len(p) > 1 else np.nan
    }

# Main Loop
all_results = {}

for domain in DOMAINS:
    print(f"\n{'='*60}")
    print(f"{domain.upper()}")
    print(f"{'='*60}")

    # Load ground truth
    df_gt = load_ground_truth(DOMAIN_MAPPING[domain])
    print(f"Ground truth: {len(df_gt)} samples")

    # Load predictions
    df_pred = load_evaluations_and_calc_predictions(domain)

RMSE CALCULATION

MEDICAL_COVIDQA


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

covidqa/train-00000-of-00001.parquet:   0%|          | 0.00/4.20M [00:00<?, ?B/s]

covidqa/test-00000-of-00001.parquet:   0%|          | 0.00/854k [00:00<?, ?B/s]

...

cuad/test-00000-of-00001.parquet:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/510 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/510 [00:00<?, ? examples/s]

Ground truth: 510 samples


In [6]:
# ============================================================================
# RMSE CALCULATION (Complete)
# ============================================================================

import numpy as np
import pandas as pd
from datasets import load_dataset
import json
import os

# Configuration
DOMAINS = [
    "medical_covidqa",
    "finance_finqa",
    "support_techqa",
    "general_hotpotqa",
    "legal_cuad"
]

DOMAIN_MAPPING = {
    "medical_covidqa": "covidqa",
    "finance_finqa": "finqa",
    "support_techqa": "techqa",
    "general_hotpotqa": "hotpotqa",
    "legal_cuad": "cuad"
}

RESULTS_BASE_DIR = "/content/drive/MyDrive/ragbench_v2/results/final_multi_domain_answers"
EVALUATION_DIR = "/content/drive/MyDrive/ragbench_v2/results/evaluation_with_llm_judge"
OUTPUT_DIR = "/content/drive/MyDrive/ragbench_v2/results/rmse_corrected"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("="*80)
print("RMSE CALCULATION")
print("="*80)

# Load Ground Truth
def load_ground_truth(dataset_name):
    dataset = load_dataset("galileo-ai/ragbench", dataset_name, split="test")
    records = []
    for row in dataset:
        records.append({
            "id": str(row["id"]),
            "relevance_score_gt": row.get("relevance_score", np.nan),
            "utilization_score_gt": row.get("utilization_score", np.nan),
            "completeness_score_gt": row.get("completeness_score", np.nan),
            "adherence_score_gt": float(row.get("adherence_score", 0)),
        })
    return pd.DataFrame(records)

# Load LLM Evaluations and Calculate Predictions
def load_evaluations_and_calc_predictions(domain):
    eval_dir = os.path.join(EVALUATION_DIR, domain)
    results_dir = os.path.join(RESULTS_BASE_DIR, domain)

    predictions = []
    eval_files = [f for f in os.listdir(eval_dir) if f.endswith('_evaluation.json')]

    for eval_file in eval_files:
        q_id = eval_file.replace('_evaluation.json', '')

        try:
            # Load evaluation
            with open(os.path.join(eval_dir, eval_file), 'r') as f:
                evaluation = json.load(f)

            # Load retrieved docs to get count
            docs_file = os.path.join(results_dir, f"{q_id}_retrieved_documents.json")
            with open(docs_file, 'r') as f:
                retrieved_docs = json.load(f)

            num_retrieved = len(retrieved_docs)
            relevant_keys = set(evaluation.get("all_relevant_sentence_keys", []))
            utilized_keys = set(evaluation.get("all_utilized_sentence_keys", []))
            adherence = float(evaluation.get("overall_supported", False))

            # Calculate scores
            relevance_score = len(relevant_keys) / num_retrieved if num_retrieved > 0 else 0.0
            utilization_score = len(utilized_keys) / num_retrieved if num_retrieved > 0 else 0.0
            relevant_utilized = relevant_keys.intersection(utilized_keys)
            completeness_score = len(relevant_utilized) / len(relevant_keys) if len(relevant_keys) > 0 else 0.0

            predictions.append({
                "id": q_id,
                "relevance_score_pred": relevance_score,
                "utilization_score_pred": utilization_score,
                "completeness_score_pred": completeness_score,
                "adherence_score_pred": adherence
            })
        except Exception as e:
            print(f"  Error processing {q_id}: {e}")
            continue

    return pd.DataFrame(predictions)

# Calculate RMSE
def calc_rmse(pred, gt):
    valid = ~(np.isnan(pred) | np.isnan(gt))
    if not np.any(valid):
        return {"rmse": np.nan, "mae": np.nan, "bias": np.nan, "n": 0, "corr": np.nan}
    p, g = pred[valid], gt[valid]
    corr = float(np.corrcoef(p, g)[0, 1]) if len(p) > 1 else np.nan
    return {
        "rmse": float(np.sqrt(np.mean((p - g) ** 2))),
        "mae": float(np.mean(np.abs(p - g))),
        "bias": float(np.mean(p - g)),
        "n": int(len(p)),
        "corr": corr if not np.isnan(corr) else 0.0
    }

# Main Loop
all_results = {}

for domain in DOMAINS:
    print(f"\n{'='*60}")
    print(f"{domain.upper()}")
    print(f"{'='*60}")

    try:
        # Load ground truth
        df_gt = load_ground_truth(DOMAIN_MAPPING[domain])
        print(f"Ground truth: {len(df_gt)} samples")

        # Load predictions
        df_pred = load_evaluations_and_calc_predictions(domain)
        print(f"Predictions: {len(df_pred)} samples")

        # Merge on ID
        df_merged = pd.merge(df_pred, df_gt, on="id", how="inner")
        print(f"Matched: {len(df_merged)} samples")

        if len(df_merged) == 0:
            print("⚠️ No matching IDs - checking ID formats...")
            print(f"  Prediction IDs: {df_pred['id'].tolist()[:3]}")
            print(f"  Ground truth IDs: {df_gt['id'].tolist()[:3]}")
            continue

        # Calculate RMSE for each metric
        results = {}
        print(f"\n{'Metric':<25} {'RMSE':<10} {'MAE':<10} {'Bias':<10} {'Corr':<10} {'N':<6}")
        print(f"{'-'*71}")

        for metric in ["relevance_score", "utilization_score", "completeness_score", "adherence_score"]:
            stats = calc_rmse(
                df_merged[f"{metric}_pred"].values,
                df_merged[f"{metric}_gt"].values
            )
            results[metric] = stats
            print(f"{metric:<25} {stats['rmse']:<10.4f} {stats['mae']:<10.4f} "
                  f"{stats['bias']:<10.4f} {stats['corr']:<10.4f} {stats['n']:<6}")

        # Overall RMSE (excluding adherence)
        rmses = [results[m]['rmse'] for m in ["relevance_score", "utilization_score", "completeness_score"]
                 if not np.isnan(results[m]['rmse'])]
        overall_rmse = np.mean(rmses) if rmses else np.nan
        results['overall_rmse'] = overall_rmse
        results['n_matched'] = len(df_merged)

        print(f"\n{'OVERALL RMSE':<25} {overall_rmse:.4f}")

        all_results[domain] = results

        # Save detailed results
        df_merged.to_csv(os.path.join(OUTPUT_DIR, f"{domain}_predictions_vs_gt.csv"), index=False)

    except Exception as e:
        print(f"Error processing {domain}: {e}")
        continue

# Final Summary
print(f"\n{'='*80}")
print("FINAL SUMMARY")
print(f"{'='*80}")
print(f"\n{'Domain':<25} {'RMSE':<12} {'N':<10}")
print(f"{'-'*47}")

domain_rmses = []
for domain, results in all_results.items():
    rmse = results['overall_rmse']
    n = results['n_matched']
    print(f"{domain:<25} {rmse:<12.4f} {n:<10}")
    if not np.isnan(rmse):
        domain_rmses.append(rmse)

avg_rmse = np.mean(domain_rmses) if domain_rmses else np.nan
print(f"\n{'AVERAGE RMSE':<25} {avg_rmse:<12.4f}")

# Save summary
summary = {"per_domain": all_results, "average_rmse": float(avg_rmse)}
with open(os.path.join(OUTPUT_DIR, "rmse_summary.json"), 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(f"\n✅ Results saved to: {OUTPUT_DIR}")

RMSE CALCULATION

MEDICAL_COVIDQA
Ground truth: 246 samples
Predictions: 30 samples
Matched: 0 samples
⚠️ No matching IDs - checking ID formats...
  Prediction IDs: ['358', '457', '1172']
  Ground truth IDs: ['1421', '677', '39']

FINANCE_FINQA
Ground truth: 2294 samples
Predictions: 30 samples
Matched: 0 samples
⚠️ No matching IDs - checking ID formats...
  Prediction IDs: ['finqa_2311', 'finqa_5136', 'finqa_664']
  Ground truth IDs: ['finqa_6345', 'finqa_7055', 'finqa_6634']

SUPPORT_TECHQA
Ground truth: 314 samples
Predictions: 30 samples
Matched: 0 samples
⚠️ No matching IDs - checking ID formats...
  Prediction IDs: ['techqa_TRAIN_Q337', 'techqa_TRAIN_Q430', 'techqa_TRAIN_Q479']
  Ground truth IDs: ['techqa_DEV_Q243', 'techqa_DEV_Q280', 'techqa_DEV_Q253']

GENERAL_HOTPOTQA
Ground truth: 390 samples
Predictions: 30 samples
Matched: 0 samples
⚠️ No matching IDs - checking ID formats...
  Prediction IDs: ['5a732ab25542991f29ee2d25', '5abf8e135542990832d3a14f', '5a89bbb05542992e4fca83

In [7]:
# ============================================================================
# RMSE CALCULATION - SEARCH ALL SPLITS
# ============================================================================

import numpy as np
import pandas as pd
from datasets import load_dataset
import json
import os

# Configuration
DOMAINS = [
    "medical_covidqa",
    "finance_finqa",
    "support_techqa",
    "general_hotpotqa",
    "legal_cuad"
]

DOMAIN_MAPPING = {
    "medical_covidqa": "covidqa",
    "finance_finqa": "finqa",
    "support_techqa": "techqa",
    "general_hotpotqa": "hotpotqa",
    "legal_cuad": "cuad"
}

RESULTS_BASE_DIR = "/content/drive/MyDrive/ragbench_v2/results/final_multi_domain_answers"
EVALUATION_DIR = "/content/drive/MyDrive/ragbench_v2/results/evaluation_with_llm_judge"
OUTPUT_DIR = "/content/drive/MyDrive/ragbench_v2/results/rmse_corrected"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("="*80)
print("RMSE CALCULATION - SEARCHING ALL SPLITS")
print("="*80)

# Load Ground Truth from ALL splits
def load_ground_truth_all_splits(dataset_name):
    all_records = []

    for split in ["train", "validation", "test"]:
        try:
            dataset = load_dataset("galileo-ai/ragbench", dataset_name, split=split)
            for row in dataset:
                all_records.append({
                    "id": str(row["id"]),
                    "split": split,
                    "relevance_score_gt": row.get("relevance_score", np.nan),
                    "utilization_score_gt": row.get("utilization_score", np.nan),
                    "completeness_score_gt": row.get("completeness_score", np.nan),
                    "adherence_score_gt": float(row.get("adherence_score", 0)),
                })
        except Exception as e:
            print(f"  Could not load {split}: {e}")
            continue

    return pd.DataFrame(all_records)

# Load LLM Evaluations and Calculate Predictions
def load_evaluations_and_calc_predictions(domain):
    eval_dir = os.path.join(EVALUATION_DIR, domain)
    results_dir = os.path.join(RESULTS_BASE_DIR, domain)

    predictions = []
    eval_files = [f for f in os.listdir(eval_dir) if f.endswith('_evaluation.json')]

    for eval_file in eval_files:
        q_id = eval_file.replace('_evaluation.json', '')

        try:
            with open(os.path.join(eval_dir, eval_file), 'r') as f:
                evaluation = json.load(f)

            docs_file = os.path.join(results_dir, f"{q_id}_retrieved_documents.json")
            with open(docs_file, 'r') as f:
                retrieved_docs = json.load(f)

            num_retrieved = len(retrieved_docs)
            relevant_keys = set(evaluation.get("all_relevant_sentence_keys", []))
            utilized_keys = set(evaluation.get("all_utilized_sentence_keys", []))
            adherence = float(evaluation.get("overall_supported", False))

            relevance_score = len(relevant_keys) / num_retrieved if num_retrieved > 0 else 0.0
            utilization_score = len(utilized_keys) / num_retrieved if num_retrieved > 0 else 0.0
            relevant_utilized = relevant_keys.intersection(utilized_keys)
            completeness_score = len(relevant_utilized) / len(relevant_keys) if len(relevant_keys) > 0 else 0.0

            predictions.append({
                "id": q_id,
                "relevance_score_pred": relevance_score,
                "utilization_score_pred": utilization_score,
                "completeness_score_pred": completeness_score,
                "adherence_score_pred": adherence
            })
        except Exception as e:
            continue

    return pd.DataFrame(predictions)

# Calculate RMSE
def calc_rmse(pred, gt):
    valid = ~(np.isnan(pred) | np.isnan(gt))
    if not np.any(valid):
        return {"rmse": np.nan, "mae": np.nan, "bias": np.nan, "n": 0, "corr": np.nan}
    p, g = pred[valid], gt[valid]
    corr = float(np.corrcoef(p, g)[0, 1]) if len(p) > 1 else np.nan
    return {
        "rmse": float(np.sqrt(np.mean((p - g) ** 2))),
        "mae": float(np.mean(np.abs(p - g))),
        "bias": float(np.mean(p - g)),
        "n": int(len(p)),
        "corr": corr if not np.isnan(corr) else 0.0
    }

# Main Loop
all_results = {}

for domain in DOMAINS:
    print(f"\n{'='*60}")
    print(f"{domain.upper()}")
    print(f"{'='*60}")

    try:
        # Load ground truth from ALL splits
        print("Loading ground truth from all splits...")
        df_gt = load_ground_truth_all_splits(DOMAIN_MAPPING[domain])
        print(f"Ground truth total: {len(df_gt)} samples")

        # Load predictions
        df_pred = load_evaluations_and_calc_predictions(domain)
        print(f"Predictions: {len(df_pred)} samples")

        # Merge on ID
        df_merged = pd.merge(df_pred, df_gt, on="id", how="inner")
        print(f"Matched: {len(df_merged)} samples")

        if len(df_merged) > 0:
            # Show which splits matched
            split_counts = df_merged['split'].value_counts()
            print(f"Splits matched: {dict(split_counts)}")

        if len(df_merged) == 0:
            print("⚠️ Still no matches!")
            continue

        # Calculate RMSE
        results = {}
        print(f"\n{'Metric':<25} {'RMSE':<10} {'MAE':<10} {'Bias':<10} {'Corr':<10} {'N':<6}")
        print(f"{'-'*71}")

        for metric in ["relevance_score", "utilization_score", "completeness_score", "adherence_score"]:
            stats = calc_rmse(
                df_merged[f"{metric}_pred"].values,
                df_merged[f"{metric}_gt"].values
            )
            results[metric] = stats
            print(f"{metric:<25} {stats['rmse']:<10.4f} {stats['mae']:<10.4f} "
                  f"{stats['bias']:<10.4f} {stats['corr']:<10.4f} {stats['n']:<6}")

        # Overall RMSE
        rmses = [results[m]['rmse'] for m in ["relevance_score", "utilization_score", "completeness_score"]
                 if not np.isnan(results[m]['rmse'])]
        overall_rmse = np.mean(rmses) if rmses else np.nan
        results['overall_rmse'] = overall_rmse
        results['n_matched'] = len(df_merged)

        print(f"\n{'OVERALL RMSE':<25} {overall_rmse:.4f}")

        all_results[domain] = results
        df_merged.to_csv(os.path.join(OUTPUT_DIR, f"{domain}_predictions_vs_gt.csv"), index=False)

    except Exception as e:
        print(f"Error: {e}")
        continue

# Final Summary
print(f"\n{'='*80}")
print("FINAL SUMMARY")
print(f"{'='*80}")
print(f"\n{'Domain':<25} {'RMSE':<12} {'N':<10}")
print(f"{'-'*47}")

domain_rmses = []
for domain, results in all_results.items():
    rmse = results['overall_rmse']
    n = results['n_matched']
    print(f"{domain:<25} {rmse:<12.4f} {n:<10}")
    if not np.isnan(rmse):
        domain_rmses.append(rmse)

avg_rmse = np.mean(domain_rmses) if domain_rmses else np.nan
print(f"\n{'AVERAGE RMSE':<25} {avg_rmse:<12.4f}")

# Save
summary = {"per_domain": all_results, "average_rmse": float(avg_rmse)}
with open(os.path.join(OUTPUT_DIR, "rmse_summary.json"), 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(f"\n✅ Results saved to: {OUTPUT_DIR}")

RMSE CALCULATION - SEARCHING ALL SPLITS

MEDICAL_COVIDQA
Loading ground truth from all splits...
Ground truth total: 1765 samples
Predictions: 30 samples
Matched: 30 samples
Splits matched: {'train': np.int64(30)}

Metric                    RMSE       MAE        Bias       Corr       N     
-----------------------------------------------------------------------
relevance_score           0.2968     0.2455     0.1835     0.2744     30    
utilization_score         0.3416     0.2806     0.2419     0.2849     30    
completeness_score        0.4564     0.3680     0.1562     -0.1679    30    
adherence_score           0.6583     0.4333     -0.4333    0.3651     30    

OVERALL RMSE              0.3649

FINANCE_FINQA
Loading ground truth from all splits...
Ground truth total: 16562 samples
Predictions: 30 samples
Matched: 60 samples
Splits matched: {'train': np.int64(60)}

Metric                    RMSE       MAE        Bias       Corr       N     
-------------------------------------------

In [11]:
# ============================================================================
# DIAGNOSTIC: Check Multiple Samples for Key Patterns
# ============================================================================

import json
import os
from datasets import load_dataset

print("="*80)
print("CHECKING RAGBENCH KEY PATTERNS ACROSS SAMPLES")
print("="*80)

# Check multiple datasets
for dataset_name in ["covidqa", "finqa", "hotpotqa"]:
    print(f"\n{'='*60}")
    print(f"DATASET: {dataset_name}")
    print(f"{'='*60}")

    dataset = load_dataset("galileo-ai/ragbench", dataset_name, split="train")

    # Count how many have non-empty keys
    has_relevant = 0
    has_utilized = 0
    total = min(100, len(dataset))

    sample_with_keys = None

    for i, row in enumerate(dataset):
        if i >= total:
            break

        rel_keys = row.get('relevant_sentence_keys', [])
        util_keys = row.get('utilized_sentence_keys', [])

        if rel_keys:
            has_relevant += 1
            if sample_with_keys is None:
                sample_with_keys = row
        if util_keys:
            has_utilized += 1

    print(f"Checked: {total} samples")
    print(f"Has relevant_sentence_keys: {has_relevant}/{total}")
    print(f"Has utilized_sentence_keys: {has_utilized}/{total}")

    if sample_with_keys:
        print(f"\nSample with keys (ID: {sample_with_keys['id']}):")
        print(f"  relevant_sentence_keys: {sample_with_keys.get('relevant_sentence_keys', [])[:5]}")
        print(f"  utilized_sentence_keys: {sample_with_keys.get('utilized_sentence_keys', [])[:5]}")

# Also check what columns exist
print(f"\n{'='*60}")
print("AVAILABLE COLUMNS IN RAGBENCH")
print(f"{'='*60}")
dataset = load_dataset("galileo-ai/ragbench", "covidqa", split="train")
print(f"Columns: {dataset.column_names}")

# Check one full sample
print(f"\n{'='*60}")
print("FULL SAMPLE STRUCTURE")
print(f"{'='*60}")
sample = dataset[0]
for key, value in sample.items():
    if isinstance(value, list):
        print(f"{key}: list with {len(value)} items")
        if len(value) > 0:
            print(f"  First item type: {type(value[0])}")
            if isinstance(value[0], str) and len(value[0]) < 100:
                print(f"  Sample: {value[:2]}")
    elif isinstance(value, str):
        print(f"{key}: string ({len(value)} chars)")
    else:
        print(f"{key}: {type(value).__name__} = {value}")

CHECKING RAGBENCH KEY PATTERNS ACROSS SAMPLES

DATASET: covidqa
Checked: 100 samples
Has relevant_sentence_keys: 0/100
Has utilized_sentence_keys: 0/100

DATASET: finqa
Checked: 100 samples
Has relevant_sentence_keys: 0/100
Has utilized_sentence_keys: 0/100

DATASET: hotpotqa
Checked: 100 samples
Has relevant_sentence_keys: 0/100
Has utilized_sentence_keys: 0/100

AVAILABLE COLUMNS IN RAGBENCH
Columns: ['id', 'question', 'documents', 'response', 'generation_model_name', 'annotating_model_name', 'dataset_name', 'documents_sentences', 'response_sentences', 'sentence_support_information', 'unsupported_response_sentence_keys', 'adherence_score', 'overall_supported_explanation', 'relevance_explanation', 'all_relevant_sentence_keys', 'all_utilized_sentence_keys', 'trulens_groundedness', 'trulens_context_relevance', 'ragas_faithfulness', 'ragas_context_relevance', 'gpt3_adherence', 'gpt3_context_relevance', 'gpt35_utilization', 'relevance_score', 'utilization_score', 'completeness_score']

FU

In [13]:
# ============================================================================
# F1 SCORE CALCULATION - CORRECTED
# ============================================================================

import numpy as np
import pandas as pd
from datasets import load_dataset
import json
import os
import re

# Configuration
DOMAINS = [
    "medical_covidqa",
    "finance_finqa",
    "support_techqa",
    "general_hotpotqa",
    "legal_cuad"
]

DOMAIN_MAPPING = {
    "medical_covidqa": "covidqa",
    "finance_finqa": "finqa",
    "support_techqa": "techqa",
    "general_hotpotqa": "hotpotqa",
    "legal_cuad": "cuad"
}

EVALUATION_DIR = "/content/drive/MyDrive/ragbench_v2/results/evaluation_with_llm_judge"
OUTPUT_DIR = "/content/drive/MyDrive/ragbench_v2/results/f1_scores"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("="*80)
print("F1 SCORE CALCULATION - CORRECTED")
print("="*80)

def extract_doc_index_from_key(key):
    """
    Convert sentence key to document index.
    RAGBench format: '0d', '1a', '2b' -> extract doc index 0, 1, 2
    LLM format: 'doc_0', 'doc_1' -> extract doc index 0, 1
    """
    if isinstance(key, str):
        # RAGBench format: '0d', '1a', '2b' (number + letter)
        match = re.match(r'^(\d+)[a-z]$', key)
        if match:
            return int(match.group(1))

        # LLM format: 'doc_0', 'doc_1'
        match = re.match(r'^doc_(\d+)$', key)
        if match:
            return int(match.group(1))

        # Try plain number
        if key.isdigit():
            return int(key)

    return None

def keys_to_doc_indices(keys):
    """Convert list of keys to set of document indices."""
    doc_indices = set()
    for key in keys:
        idx = extract_doc_index_from_key(key)
        if idx is not None:
            doc_indices.add(idx)
    return doc_indices

# Load Ground Truth from ALL splits with non-empty keys
def load_ground_truth_with_keys(dataset_name):
    all_records = []
    for split in ["train", "validation", "test"]:
        try:
            dataset = load_dataset("galileo-ai/ragbench", dataset_name, split=split)
            for row in dataset:
                rel_keys = row.get('all_relevant_sentence_keys', [])
                util_keys = row.get('all_utilized_sentence_keys', [])

                # Only include samples with non-empty keys
                if rel_keys or util_keys:
                    all_records.append({
                        "id": str(row["id"]),
                        "split": split,
                        "relevant_keys_gt": rel_keys,
                        "utilized_keys_gt": util_keys,
                        "relevant_docs_gt": keys_to_doc_indices(rel_keys),
                        "utilized_docs_gt": keys_to_doc_indices(util_keys),
                    })
        except:
            continue
    return all_records

# Load LLM Evaluations
def load_evaluations(domain):
    eval_dir = os.path.join(EVALUATION_DIR, domain)
    evaluations = {}

    for eval_file in os.listdir(eval_dir):
        if eval_file.endswith('_evaluation.json'):
            q_id = eval_file.replace('_evaluation.json', '')
            with open(os.path.join(eval_dir, eval_file), 'r') as f:
                evaluation = json.load(f)

            rel_keys = evaluation.get("all_relevant_sentence_keys", [])
            util_keys = evaluation.get("all_utilized_sentence_keys", [])

            evaluations[q_id] = {
                "relevant_keys_pred": rel_keys,
                "utilized_keys_pred": util_keys,
                "relevant_docs_pred": keys_to_doc_indices(rel_keys),
                "utilized_docs_pred": keys_to_doc_indices(util_keys),
            }

    return evaluations

def calculate_f1(precision, recall):
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

# Main Loop
all_results = {}

for domain in DOMAINS:
    print(f"\n{'='*60}")
    print(f"{domain.upper()}")
    print(f"{'='*60}")

    try:
        # Load ground truth (only samples with keys)
        gt_list = load_ground_truth_with_keys(DOMAIN_MAPPING[domain])
        gt_dict = {r['id']: r for r in gt_list}
        print(f"Ground truth with keys: {len(gt_list)} samples")

        # Load evaluations
        evaluations = load_evaluations(domain)
        print(f"Evaluations: {len(evaluations)} samples")

        # Find matches
        matched_ids = set(evaluations.keys()).intersection(set(gt_dict.keys()))
        print(f"Matched (with GT keys): {len(matched_ids)} samples")

        if len(matched_ids) == 0:
            print("⚠️ No matches with annotated ground truth")
            print("   Calculating F1 using LLM self-consistency instead...")

            # Alternative: Calculate internal consistency metrics
            f1_results = []
            for q_id, eval_data in evaluations.items():
                pred_rel = eval_data['relevant_docs_pred']
                pred_util = eval_data['utilized_docs_pred']

                # Self-consistency: utilized should be subset of relevant
                if len(pred_rel) > 0:
                    consistency = len(pred_util.intersection(pred_rel)) / len(pred_rel)
                else:
                    consistency = 0

                f1_results.append({
                    "id": q_id,
                    "n_relevant_pred": len(pred_rel),
                    "n_utilized_pred": len(pred_util),
                    "consistency": consistency
                })

            df_f1 = pd.DataFrame(f1_results)
            print(f"\nSelf-Consistency Metrics:")
            print(f"  Avg relevant docs identified: {df_f1['n_relevant_pred'].mean():.2f}")
            print(f"  Avg utilized docs identified: {df_f1['n_utilized_pred'].mean():.2f}")
            print(f"  Avg consistency: {df_f1['consistency'].mean():.4f}")

            all_results[domain] = {
                "n_samples": len(evaluations),
                "matched_with_gt": 0,
                "avg_relevant_docs": float(df_f1['n_relevant_pred'].mean()),
                "avg_utilized_docs": float(df_f1['n_utilized_pred'].mean()),
                "consistency": float(df_f1['consistency'].mean()),
                "note": "No GT keys available - showing self-consistency"
            }
            continue

        # Calculate F1 for matched samples
        f1_results = []

        for q_id in matched_ids:
            gt = gt_dict[q_id]
            pred = evaluations[q_id]

            # Document-level comparison
            gt_rel_docs = gt['relevant_docs_gt']
            gt_util_docs = gt['utilized_docs_gt']
            pred_rel_docs = pred['relevant_docs_pred']
            pred_util_docs = pred['utilized_docs_pred']

            # Relevance F1 (document level)
            if len(gt_rel_docs) > 0:
                rel_tp = len(pred_rel_docs.intersection(gt_rel_docs))
                rel_precision = rel_tp / len(pred_rel_docs) if len(pred_rel_docs) > 0 else 0
                rel_recall = rel_tp / len(gt_rel_docs)
                rel_f1 = calculate_f1(rel_precision, rel_recall)
            else:
                rel_precision, rel_recall, rel_f1 = 0, 0, 0

            # Utilization F1 (document level)
            if len(gt_util_docs) > 0:
                util_tp = len(pred_util_docs.intersection(gt_util_docs))
                util_precision = util_tp / len(pred_util_docs) if len(pred_util_docs) > 0 else 0
                util_recall = util_tp / len(gt_util_docs)
                util_f1 = calculate_f1(util_precision, util_recall)
            else:
                util_precision, util_recall, util_f1 = 0, 0, 0

            f1_results.append({
                "id": q_id,
                "relevance_precision": rel_precision,
                "relevance_recall": rel_recall,
                "relevance_f1": rel_f1,
                "utilization_precision": util_precision,
                "utilization_recall": util_recall,
                "utilization_f1": util_f1,
            })

        df_f1 = pd.DataFrame(f1_results)

        # Calculate averages
        print(f"\n{'Metric':<25} {'Precision':<12} {'Recall':<12} {'F1':<12}")
        print(f"{'-'*61}")
        print(f"{'Relevance':<25} {df_f1['relevance_precision'].mean():<12.4f} "
              f"{df_f1['relevance_recall'].mean():<12.4f} {df_f1['relevance_f1'].mean():<12.4f}")
        print(f"{'Utilization':<25} {df_f1['utilization_precision'].mean():<12.4f} "
              f"{df_f1['utilization_recall'].mean():<12.4f} {df_f1['utilization_f1'].mean():<12.4f}")

        overall_f1 = (df_f1['relevance_f1'].mean() + df_f1['utilization_f1'].mean()) / 2
        print(f"\n{'OVERALL F1':<25} {overall_f1:.4f}")

        all_results[domain] = {
            "relevance_f1": float(df_f1['relevance_f1'].mean()),
            "utilization_f1": float(df_f1['utilization_f1'].mean()),
            "overall_f1": float(overall_f1),
            "n_samples": len(df_f1),
            "matched_with_gt": len(matched_ids)
        }

        df_f1.to_csv(os.path.join(OUTPUT_DIR, f"{domain}_f1_scores.csv"), index=False)

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()
        continue

# Final Summary
print(f"\n{'='*80}")
print("FINAL F1 SUMMARY")
print(f"{'='*80}")

for domain, results in all_results.items():
    print(f"\n{domain}:")
    if results.get('matched_with_gt', 0) > 0:
        print(f"  Relevance F1: {results['relevance_f1']:.4f}")
        print(f"  Utilization F1: {results['utilization_f1']:.4f}")
        print(f"  Overall F1: {results['overall_f1']:.4f}")
        print(f"  Samples: {results['n_samples']}")
    else:
        print(f"  No GT keys - Self-consistency: {results.get('consistency', 'N/A')}")
        print(f"  Avg relevant docs: {results.get('avg_relevant_docs', 'N/A')}")
        print(f"  Avg utilized docs: {results.get('avg_utilized_docs', 'N/A')}")

# Save
with open(os.path.join(OUTPUT_DIR, "f1_summary.json"), 'w') as f:
    json.dump(all_results, f, indent=2, default=str)

print(f"\n✅ Results saved to: {OUTPUT_DIR}")

F1 SCORE CALCULATION - CORRECTED

MEDICAL_COVIDQA
Ground truth with keys: 1734 samples
Evaluations: 30 samples
Matched (with GT keys): 28 samples

Metric                    Precision    Recall       F1          
-------------------------------------------------------------
Relevance                 0.3410       0.7024       0.4343      
Utilization               0.3107       0.5417       0.3502      

OVERALL F1                0.3922

FINANCE_FINQA
Ground truth with keys: 16518 samples
Evaluations: 30 samples
Matched (with GT keys): 30 samples

Metric                    Precision    Recall       F1          
-------------------------------------------------------------
Relevance                 0.0000       0.0000       0.0000      
Utilization               0.0000       0.0000       0.0000      

OVERALL F1                0.0000

SUPPORT_TECHQA
Ground truth with keys: 1614 samples
Evaluations: 30 samples
Matched (with GT keys): 29 samples

Metric                    Precision    Recall

In [14]:
# ============================================================================
# ADHERENCE AUROC & F1 CALCULATION
# ============================================================================
# Per RAGBench paper (Table 3): Report AUROC for hallucination detection
# Adherence = whether response is fully grounded in context (no hallucinations)

import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score
import json
import os

# Configuration
DOMAINS = [
    "medical_covidqa",
    "finance_finqa",
    "support_techqa",
    "general_hotpotqa",
    "legal_cuad"
]

DOMAIN_MAPPING = {
    "medical_covidqa": "covidqa",
    "finance_finqa": "finqa",
    "support_techqa": "techqa",
    "general_hotpotqa": "hotpotqa",
    "legal_cuad": "cuad"
}

EVALUATION_DIR = "/content/drive/MyDrive/ragbench_v2/results/evaluation_with_llm_judge"
OUTPUT_DIR = "/content/drive/MyDrive/ragbench_v2/results/adherence_metrics"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("="*80)
print("ADHERENCE AUROC & F1 CALCULATION")
print("="*80)

# Load Ground Truth from ALL splits
def load_ground_truth_all_splits(dataset_name):
    """Load adherence ground truth from all splits."""
    all_records = {}
    for split in ["train", "validation", "test"]:
        try:
            dataset = load_dataset("galileo-ai/ragbench", dataset_name, split=split)
            for row in dataset:
                q_id = str(row["id"])
                all_records[q_id] = {
                    "adherence_gt": row.get("adherence_score", None),  # Boolean
                    "split": split
                }
        except Exception as e:
            print(f"  Could not load {split}: {e}")
            continue
    return all_records

# Load LLM Evaluations
def load_evaluations(domain):
    """Load LLM evaluation results."""
    eval_dir = os.path.join(EVALUATION_DIR, domain)
    evaluations = {}

    for eval_file in os.listdir(eval_dir):
        if eval_file.endswith('_evaluation.json'):
            q_id = eval_file.replace('_evaluation.json', '')
            filepath = os.path.join(eval_dir, eval_file)

            with open(filepath, 'r') as f:
                evaluation = json.load(f)

            # Get predicted adherence (overall_supported)
            overall_supported = evaluation.get("overall_supported", None)

            evaluations[q_id] = {
                "adherence_pred": overall_supported,  # Boolean
            }

    return evaluations

# Main calculation
all_results = {}
all_y_true = []
all_y_pred = []

for domain in DOMAINS:
    print(f"\n{'='*60}")
    print(f"{domain.upper()}")
    print(f"{'='*60}")

    try:
        # Load ground truth
        gt_dict = load_ground_truth_all_splits(DOMAIN_MAPPING[domain])
        print(f"Ground truth loaded: {len(gt_dict)} samples")

        # Load evaluations
        evaluations = load_evaluations(domain)
        print(f"Evaluations loaded: {len(evaluations)} samples")

        # Find matches
        matched_ids = set(evaluations.keys()).intersection(set(gt_dict.keys()))
        print(f"Matched: {len(matched_ids)} samples")

        if len(matched_ids) == 0:
            print("⚠️ No matches found!")
            continue

        # Collect predictions and ground truth
        y_true = []  # Ground truth adherence (1 = supported, 0 = hallucinated)
        y_pred = []  # Predicted adherence

        for q_id in matched_ids:
            gt_adherence = gt_dict[q_id]["adherence_gt"]
            pred_adherence = evaluations[q_id]["adherence_pred"]

            # Skip if either is None
            if gt_adherence is None or pred_adherence is None:
                continue

            # Convert to int (1 = adherent/supported, 0 = hallucinated)
            y_true.append(int(gt_adherence))
            y_pred.append(int(pred_adherence))

        y_true = np.array(y_true)
        y_pred = np.array(y_pred)

        print(f"Valid samples: {len(y_true)}")
        print(f"GT distribution: {np.sum(y_true)} adherent, {len(y_true) - np.sum(y_true)} hallucinated")
        print(f"Pred distribution: {np.sum(y_pred)} adherent, {len(y_pred) - np.sum(y_pred)} hallucinated")

        # Calculate metrics
        # Note: AUROC requires both classes present
        if len(np.unique(y_true)) < 2:
            print("⚠️ Only one class in ground truth - cannot compute AUROC")
            auroc = None
        else:
            auroc = roc_auc_score(y_true, y_pred)

        # F1, Precision, Recall, Accuracy
        f1 = f1_score(y_true, y_pred, zero_division=0)
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        accuracy = accuracy_score(y_true, y_pred)

        # For hallucination detection, we often report metrics on the hallucination class
        # (inverted: predicting hallucinations)
        y_true_hal = 1 - y_true  # 1 = hallucinated, 0 = adherent
        y_pred_hal = 1 - y_pred

        if len(np.unique(y_true_hal)) >= 2:
            auroc_hal = roc_auc_score(y_true_hal, y_pred_hal)
        else:
            auroc_hal = auroc  # Same as adherent AUROC

        f1_hal = f1_score(y_true_hal, y_pred_hal, zero_division=0)

        print(f"\n{'Metric':<25} {'Value':<12}")
        print(f"{'-'*37}")
        print(f"{'AUROC (Adherence)':<25} {auroc if auroc else 'N/A':<12}")
        print(f"{'AUROC (Hallucination)':<25} {auroc_hal if auroc_hal else 'N/A':<12}")
        print(f"{'F1 (Adherence)':<25} {f1:.4f}")
        print(f"{'F1 (Hallucination)':<25} {f1_hal:.4f}")
        print(f"{'Precision':<25} {precision:.4f}")
        print(f"{'Recall':<25} {recall:.4f}")
        print(f"{'Accuracy':<25} {accuracy:.4f}")

        all_results[domain] = {
            "auroc": float(auroc) if auroc else None,
            "auroc_hallucination": float(auroc_hal) if auroc_hal else None,
            "f1_adherence": float(f1),
            "f1_hallucination": float(f1_hal),
            "precision": float(precision),
            "recall": float(recall),
            "accuracy": float(accuracy),
            "n_samples": len(y_true),
            "n_adherent_gt": int(np.sum(y_true)),
            "n_hallucinated_gt": int(len(y_true) - np.sum(y_true))
        }

        # Aggregate for overall calculation
        all_y_true.extend(y_true.tolist())
        all_y_pred.extend(y_pred.tolist())

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()
        continue

# Overall metrics across all domains
print(f"\n{'='*80}")
print("OVERALL METRICS (All Domains Combined)")
print(f"{'='*80}")

all_y_true = np.array(all_y_true)
all_y_pred = np.array(all_y_pred)

print(f"Total samples: {len(all_y_true)}")
print(f"GT distribution: {np.sum(all_y_true)} adherent, {len(all_y_true) - np.sum(all_y_true)} hallucinated")

if len(np.unique(all_y_true)) >= 2:
    overall_auroc = roc_auc_score(all_y_true, all_y_pred)
    overall_auroc_hal = roc_auc_score(1 - all_y_true, 1 - all_y_pred)
else:
    overall_auroc = None
    overall_auroc_hal = None

overall_f1 = f1_score(all_y_true, all_y_pred, zero_division=0)
overall_f1_hal = f1_score(1 - all_y_true, 1 - all_y_pred, zero_division=0)
overall_accuracy = accuracy_score(all_y_true, all_y_pred)

print(f"\n{'Metric':<30} {'Value':<12}")
print(f"{'-'*42}")
print(f"{'AUROC (Hallucination Det.)':<30} {overall_auroc_hal if overall_auroc_hal else 'N/A'}")
print(f"{'F1 (Adherence)':<30} {overall_f1:.4f}")
print(f"{'F1 (Hallucination)':<30} {overall_f1_hal:.4f}")
print(f"{'Accuracy':<30} {overall_accuracy:.4f}")

# Summary table
print(f"\n{'='*80}")
print("SUMMARY TABLE (Per RAGBench Paper Format)")
print(f"{'='*80}")
print(f"\n{'Domain':<20} {'AUROC↑':<12} {'F1':<12} {'Accuracy':<12} {'N':<8}")
print(f"{'-'*64}")
for domain, results in all_results.items():
    auroc_str = f"{results['auroc']:.4f}" if results['auroc'] else "N/A"
    print(f"{domain:<20} {auroc_str:<12} {results['f1_adherence']:.4f}       {results['accuracy']:.4f}       {results['n_samples']:<8}")

print(f"{'-'*64}")
overall_auroc_str = f"{overall_auroc:.4f}" if overall_auroc else "N/A"
print(f"{'OVERALL':<20} {overall_auroc_str:<12} {overall_f1:.4f}       {overall_accuracy:.4f}       {len(all_y_true):<8}")

# Save results
all_results["OVERALL"] = {
    "auroc": float(overall_auroc) if overall_auroc else None,
    "f1_adherence": float(overall_f1),
    "f1_hallucination": float(overall_f1_hal),
    "accuracy": float(overall_accuracy),
    "n_samples": len(all_y_true)
}

with open(os.path.join(OUTPUT_DIR, "adherence_metrics.json"), 'w') as f:
    json.dump(all_results, f, indent=2)

print(f"\n✅ Results saved to: {OUTPUT_DIR}/adherence_metrics.json")

ADHERENCE AUROC & F1 CALCULATION

MEDICAL_COVIDQA
Ground truth loaded: 1765 samples
Evaluations loaded: 30 samples
Matched: 30 samples
Valid samples: 30
GT distribution: 25 adherent, 5 hallucinated
Pred distribution: 12 adherent, 18 hallucinated

Metric                    Value       
-------------------------------------
AUROC (Adherence)         0.74        
AUROC (Hallucination)     0.74        
F1 (Adherence)            0.6486
F1 (Hallucination)        0.4348
Precision                 1.0000
Recall                    0.4800
Accuracy                  0.5667

FINANCE_FINQA
Ground truth loaded: 8281 samples
Evaluations loaded: 30 samples
Matched: 30 samples
Valid samples: 30
GT distribution: 29 adherent, 1 hallucinated
Pred distribution: 25 adherent, 5 hallucinated

Metric                    Value       
-------------------------------------
AUROC (Adherence)         0.9310344827586207
AUROC (Hallucination)     0.9310344827586207
F1 (Adherence)            0.9259
F1 (Hallucination)    